# Notebook 5: DataFrames (pandas)

> This notebook is intentionally a **light introduction**,  just enough pandas to be comfortable reading and manipulating a table. You can go much deeper using pandas (table transformation, merging).

---

## 0. Recap & motivation

NumPy arrays are great for numerical data, but real-world data usually looks like a **table**: rows and columns, often with column names, and a mix of types (numbers, text, dates...). That's exactly what [pandas](https://pandas.pydata.org/) is built for.

Pandas has two main objects:
- A **Series**: a single labeled column of data (built on top of a NumPy array).
- A **DataFrame**: a full table made of several Series, sharing a common index.

![Series and Dataframe illustration](../resources/series-and-dataframe.png "Two Series and a Dataframe")

In [ ]:
import pandas as pd   # "pd" is the near-universal convention

---

## 1. Creating a DataFrame

The easiest way to create a small DataFrame by hand is from a dictionary, where each key becomes a column:

In [ ]:
data = {
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28],
    "city": ["Paris", "Lyon", "Paris", "Marseille"]
}

df = pd.DataFrame(data)
print(df)

In practice, you'll almost always load data from a file instead of typing it by hand, most commonly with `pd.read_csv("filename.csv")`. We'll cover file loading properly in notebook 6; for this notebook, we'll keep working with small DataFrames built directly in code so we can focus on pandas itself.

A single column of a DataFrame is a **Series**:

In [ ]:
data = {"name": ["Alice", "Bob", "Charlie"], "age": [25, 31, 22]}
df = pd.DataFrame(data)

ages = df["age"]
print(ages)
print(type(ages))

### Exercise 1: Creating a DataFrame

Create a DataFrame called `products` with columns `"name"` and `"price"`, containing 4 products of your choice.

In [ ]:
# your code here

---

## 2. Getting a first look at a DataFrame

A handful of methods cover most of what you need to quickly understand a new dataset:

In [ ]:
data = {
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28],
    "city": ["Paris", "Lyon", "Paris", "Marseille"]
}
df = pd.DataFrame(data)

print(df.head(2))     # first 2 rows (default: first 5)
print(df.shape)       # (rows, columns)
print(df.columns)     # column names
print(df.info())      # column types, non-null counts
print(df.describe())  # quick statistics for numeric columns

`.describe()` is a nice one to remember: a one-line summary (count, mean, std, min, max, quartiles) of every numeric column, useful the moment you load any new dataset.

### Exercise 2: First look

Using the `products` DataFrame you created in Exercise 1, print its `shape` and call `.describe()` on it to see the summary statistics of the `price` column.

In [ ]:
# your code here

---

## 3. Selecting and filtering data

To select one or more columns:

In [ ]:
data = {
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28],
    "city": ["Paris", "Lyon", "Paris", "Marseille"]
}
df = pd.DataFrame(data)

print(df["name"])            # single column -> Series
print(df[["name", "age"]])   # multiple columns -> DataFrame

To select rows, pandas offers two main tools:

- `.loc[]`: select by **label** (row/column names)
- `.iloc[]`: select by **position** (like list indexing)

In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28]
})

print(df.loc[0])          # row with label 0
print(df.iloc[0])         # row at position 0 (same result here, since the index is 0,1,2...)
print(df.loc[1, "name"])  # value at row label 1, column "name"
print(df.iloc[0:2])       # first two rows

And just like with NumPy arrays, you can filter rows with a boolean condition:

In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28]
})

print(df[df["age"] > 25])   # only rows where age is greater than 25

### Exercise 3: Selecting & filtering

Using the `products` DataFrame from Exercise 1, print only the products with a `price` above `50`.

In [ ]:
# your code here

---

## 4. Mask-based selection with `.loc`

The boolean filtering you just saw (`df[df["age"] > 25]`) gives you back matching **rows**, with all columns. But you can combine a boolean mask with `.loc[]` to also choose **which columns** to keep in one line, instead of filtering and then selecting separately.

In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28],
    "city": ["Paris", "Lyon", "Paris", "Marseille"]
})

mask = df["age"] > 25
print(mask)   # a Series of True/False, same length as df

print(df.loc[mask])                 # same as df[mask]: matching rows, all columns
print(df.loc[mask, "name"])         # matching rows, single column -> Series
print(df.loc[mask, ["name", "city"]])  # matching rows, several columns -> DataFrame

You can also write the mask directly inside `.loc[]` without a separate variable, this is the form you'll see most often in real code:

In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28],
    "city": ["Paris", "Lyon", "Paris", "Marseille"]
})

print(df.loc[df["city"] == "Paris", "name"])

`.loc[]` also lets you **update** values for only the matching rows,  something plain boolean filtering (`df[mask]`) can't safely do since it returns a copy rather than a "live" view into the original DataFrame:

In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28]
})

df.loc[df["age"] < 25, "age"] = 25   # bump up any age below 25
print(df)

### Exercise 4: Mask-based selection with `.loc`

Using the `products` DataFrame from Exercise 1, use `.loc[]` with a boolean mask to print only the `"name"` column for products priced above `50`. Then, still using `.loc[]`, set the price of any product below `50` to `50` (a "price floor").

In [ ]:
# your code here

---

## 5. Grouping rows

`groupby` lets you split rows into groups based on a column, then compute something per group. If you're familiar with database management, it is very similar in spirit to a pivot table.

In [ ]:
df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Dan"],
    "age": [25, 31, 22, 28],
    "city": ["Paris", "Lyon", "Paris", "Marseille"]
})

print(df.groupby("city")["age"].mean())

This reads as: "group the rows by `city`, then compute the average `age` within each group". You'll use this pattern constantly once you're doing real analysis. For now, just recognize the shape of it: `df.groupby(column)[other_column].some_aggregation()`.

A quick related tool: `.value_counts()`, for counting how many times each value appears in a column:

In [ ]:
df = pd.DataFrame({
    "city": ["Paris", "Lyon", "Paris", "Marseille", "Paris"]
})

print(df["city"].value_counts())

### Exercise 5: `groupby`
Given the DataFrame below, group by `"department"` and print the average `"salary"` per department.

In [ ]:
df = pd.DataFrame({
    "employee": ["Alice", "Bob", "Charlie", "Dan", "Eve"],
    "department": ["Sales", "Sales", "Engineering", "Engineering", "Engineering"],
    "salary": [45000, 48000, 62000, 58000, 65000]
})

# your code here

---

## Wrap-up

In this notebook you got a **first taste** of pandas:
- The difference between a `Series` (one column) and a `DataFrame` (a full table)
- How to peek at a new dataset with `.head()`, `.shape`, `.info()`, and `.describe()`
- Selecting columns, and selecting rows with `.loc[]` / `.iloc[]` / boolean filtering
- A first look at `.groupby()` and `.value_counts()`

This is deliberately just the surface, things like merging tables, handling missing data in depth, and more advanced reshaping will come back in your data science course, where pandas will get a lot more attention.